In [18]:
import numpy as np
import numpy.linalg as linal
import scipy.io 
import matplotlib.pyplot as plt
from scipy.stats.distributions import chi2
mean_shift = lambda x : x - x.mean()

In [3]:
data = scipy.io.loadmat('DAEdata.mat')

In [4]:
data

{'__header__': b'MATLAB 5.0 MAT-file, Platform: GLNXA64, Created on: Sat May  6 03:43:28 2023',
 '__version__': '1.0',
 '__globals__': [],
 'stdeu': array([[1.07037733]]),
 'stdey': array([[0.10855124, 0.        , 0.        ],
        [0.        , 1.2243996 , 0.        ],
        [0.        , 0.        , 0.01745446]]),
 'umeas': array([[ 0.70959106],
        [-0.3454314 ],
        [-0.77928805],
        ...,
        [11.98035719],
        [ 0.52722613],
        [11.03769182]]),
 'ymeas': array([[ 0.84568864, -1.3449391 , -0.02169044],
        [ 0.99652317, -1.42641701, -0.03881469],
        [ 0.70186232, -1.69957051, -0.03285628],
        ...,
        [ 4.2342582 ,  7.08836225,  0.07871471],
        [ 4.27953568, -5.51891946, -0.09246718],
        [ 4.28926674,  5.05670289,  0.07189048]])}

In [8]:
Z = np.append(data['ymeas'], data['umeas'], axis=1)

In [9]:
L = np.diag(np.append(np.diag(data['stdey']), data['stdeu'][0]))

In [16]:
Zs = mean_shift(Z)@linal.inv(L)

In [17]:
U,S,V = linal.svd(Zs/np.sqrt(1024))

In [19]:
p = 4
N = 1024
d = p-1
# Y = Zm@np.linalg.inv(L.T)

# evals = np.sort(np.linalg.eig( Y.T@Y / N) [0]).real [::-1]
evals = S**2
while d > 1 : 
    n_dash = N - (2*p+11)/6
    l_dash = evals[ p-d : ].sum()/d 
    tau    = n_dash * ( d * np.log(l_dash) - np.log(evals[ p-d : ]).sum())
    fdom   = 0.5*(d+2)*(d-1)
    chi    = chi2.ppf(0.95, df=fdom)

    print(f"{d} \t||\t  TEST STATISTIC : {np.round(tau,3)} \t | \t CHI2 : {np.round(chi,3)}")

    if (tau <= chi) : 
        print()
        print("Number of Constraints :", d)
        break 
    else : 
        d = d - 1 

3 	||	  TEST STATISTIC : 2619.85 	 | 	 CHI2 : 11.07
2 	||	  TEST STATISTIC : 2379.289 	 | 	 CHI2 : 5.991


In [20]:
S**2

array([1.99308464e+04, 4.57080153e+01, 3.95050050e+01, 1.00993725e+00])